In [19]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
from itertools import combinations

In [20]:
DROPPED = ['Class', 'Time', 'Amount', 'V23', 'V27', 'V28',
           'V26', 'V25', 'V22', 'V15', 'V13', 'V24']

# Strategy:
#   'exhaustive'  – try every combination (only feasible if n_features <= ~18)
#   'stepwise'    – forward stepwise selection (fast, greedy)
#   'top_n'       – exhaustive over only the top N features by importance
TOP_N    = 12        # used when STRATEGY == 'top_n'
MIN_FEATURES = 3     # minimum subset size to test
OPTIMIZE_FOR = 'recall'   # 'recall', 'precision', or 'f1'

In [25]:
df = pd.read_csv('creditcard.csv')
ALL_FEATURES = [c for c in df.columns if c not in DROPPED]
print(f"Available features ({len(ALL_FEATURES)}): {ALL_FEATURES}\n")

X_all = df[ALL_FEATURES]
y     = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

def evaluate(features):
    model = RandomForestClassifier(n_estimators=30, class_weight='balanced', random_state=42, max_depth=20, min_samples_leaf=15)
    model.fit(X_train[list(features)], y_train)
    y_pred = model.predict(X_test[list(features)])
    return {
        'recall':    recall_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
    }

results = []

Available features (19): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21']



In [26]:
# Forward Stepwise
print("Forward stepwise selection…\n")
selected   = []
remaining  = list(ALL_FEATURES)

while remaining:
    best_score  = -1
    best_feat   = None
    for feat in remaining:
        candidate = selected + [feat]
        m = evaluate(candidate)
        score = m[OPTIMIZE_FOR]
        if len(candidate) >= MIN_FEATURES:
            results.append({'features': list(candidate), **m})
        if score > best_score:
            best_score = score
            best_feat  = feat
            best_metrics = m

    selected.append(best_feat)
    remaining.remove(best_feat)
    print(f"  Added '{best_feat}' → "
            f"recall={best_metrics['recall']:.4f}  "
            f"precision={best_metrics['precision']:.4f}  "
            f"f1={best_metrics['f1']:.4f}  "
            f"[{len(selected)} features]")

Forward stepwise selection…

  Added 'V14' → recall=0.7959  precision=0.1814  f1=0.2955  [1 features]
  Added 'V12' → recall=0.8776  precision=0.4831  f1=0.6232  [2 features]
  Added 'V2' → recall=0.8776  precision=0.6143  f1=0.7227  [3 features]
  Added 'V1' → recall=0.8776  precision=0.6143  f1=0.7227  [4 features]
  Added 'V3' → recall=0.8776  precision=0.6667  f1=0.7577  [5 features]
  Added 'V5' → recall=0.8776  precision=0.6880  f1=0.7713  [6 features]
  Added 'V6' → recall=0.8776  precision=0.7049  f1=0.7818  [7 features]
  Added 'V8' → recall=0.8673  precision=0.6967  f1=0.7727  [8 features]
  Added 'V17' → recall=0.8776  precision=0.7288  f1=0.7963  [9 features]
  Added 'V11' → recall=0.8776  precision=0.7288  f1=0.7963  [10 features]
  Added 'V16' → recall=0.8673  precision=0.7203  f1=0.7870  [11 features]
  Added 'V10' → recall=0.8776  precision=0.7478  f1=0.8075  [12 features]
  Added 'V20' → recall=0.8776  precision=0.7288  f1=0.7963  [13 features]
  Added 'V4' → recall=0.

In [29]:
results_df = pd.DataFrame(results)
results_df['n_features'] = results_df['features'].apply(len)
results_df = results_df.sort_values(OPTIMIZE_FOR, ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print(f"TOP 10 COMBINATIONS BY {OPTIMIZE_FOR.upper()}")
print("══════════════════════════════════════════════════════")
for i, row in results_df.head(10).iterrows():
    print(f"\nRank {i+1}  ({row['n_features']} features)")
    print(f"  Recall:    {row['recall']:.4f}")
    print(f"  Precision: {row['precision']:.4f}")
    print(f"  F1:        {row['f1']:.4f}")
    print(f"  Features:  {row['features']}")

# Save full results
out_path = 'random_forest_v1.csv'
results_df['features'] = results_df['features'].apply(str)
results_df.to_csv(out_path, index=False)
print(f"\nAll results saved to '{out_path}'")


══════════════════════════════════════════════════════
TOP 10 COMBINATIONS BY RECALL
══════════════════════════════════════════════════════

Rank 1  (6 features)
  Recall:    0.8776
  Precision: 0.6880
  F1:        0.7713
  Features:  ['V14', 'V12', 'V2', 'V1', 'V3', 'V5']

Rank 2  (5 features)
  Recall:    0.8776
  Precision: 0.6143
  F1:        0.7227
  Features:  ['V14', 'V12', 'V2', 'V1', 'V8']

Rank 3  (13 features)
  Recall:    0.8776
  Precision: 0.7288
  F1:        0.7963
  Features:  ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']

Rank 4  (4 features)
  Recall:    0.8776
  Precision: 0.6324
  F1:        0.7350
  Features:  ['V14', 'V12', 'V2', 'V11']

Rank 5  (4 features)
  Recall:    0.8776
  Precision: 0.6370
  F1:        0.7382
  Features:  ['V14', 'V12', 'V2', 'V16']

Rank 6  (4 features)
  Recall:    0.8776
  Precision: 0.6277
  F1:        0.7319
  Features:  ['V14', 'V12', 'V2', 'V17']

Rank 7  (4 features)
  Recall:    0.8776
  Pr

In [30]:
top_features = [
    ['V14', 'V12', 'V2', 'V1', 'V3', 'V5'],
    ['V14', 'V12', 'V2', 'V1', 'V8'],
    ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20'],
    ['V14', 'V12', 'V2', 'V11'],
    ['V14', 'V12', 'V2', 'V16'],
]

n_estimators_options = [50, 100, 300, 500]
max_depth_options    = [5, 10, 20, None] 

In [31]:
results = []

for features in top_features:
    for n_est in n_estimators_options:
        for depth in max_depth_options:
            model = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=depth,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_train[features], y_train)
            y_pred = model.predict(X_test[features])
            results.append({
                'features':     features,
                'n_estimators': n_est,
                'max_depth':    depth,
                'recall':       recall_score(y_test, y_pred),
                'precision':    precision_score(y_test, y_pred),
                'f1':           f1_score(y_test, y_pred),
            })
            print(f"features={len(features)} n_est={n_est} depth={depth} → recall={results[-1]['recall']:.4f} precision={results[-1]['precision']:.4f}")

results_df = pd.DataFrame(results).sort_values('recall', ascending=False)
print(results_df.head(10).to_string(index=False))
results_df.to_csv('tuning_results.csv', index=False)

features=6 n_est=50 depth=5 → recall=0.8878 precision=0.1590
features=6 n_est=50 depth=10 → recall=0.8469 precision=0.7345
features=6 n_est=50 depth=20 → recall=0.8061 precision=0.8681
features=6 n_est=50 depth=None → recall=0.7755 precision=0.8837
features=6 n_est=100 depth=5 → recall=0.8878 precision=0.1585
features=6 n_est=100 depth=10 → recall=0.8469 precision=0.7411
features=6 n_est=100 depth=20 → recall=0.7959 precision=0.8667
features=6 n_est=100 depth=None → recall=0.7959 precision=0.8764
features=6 n_est=300 depth=5 → recall=0.8878 precision=0.1543
features=6 n_est=300 depth=10 → recall=0.8469 precision=0.7411
features=6 n_est=300 depth=20 → recall=0.7857 precision=0.8750
features=6 n_est=300 depth=None → recall=0.7857 precision=0.8851
features=6 n_est=500 depth=5 → recall=0.8878 precision=0.1645
features=6 n_est=500 depth=10 → recall=0.8571 precision=0.7568
features=6 n_est=500 depth=20 → recall=0.7857 precision=0.8750
features=6 n_est=500 depth=None → recall=0.7857 precision

In [ ]:
fine_tuned = pd.read_csv('tuning_results.csv')

In [42]:
fine_tuned = fine_tuned.sort_values('recall', ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print("TOP FINE-TUNED MODEL BY RECALL")
print("══════════════════════════════════════════════════════")

for i, row in fine_tuned.head(1).iterrows():
    print(f"\nRank {i+1}")
    print(f"  Recall:       {row['recall']:.4f}")
    print(f"  Precision:    {row['precision']:.4f}")
    print(f"  F1:           {row['f1']:.4f}")
    print(f"  n_estimators: {row['n_estimators']}")
    print(f"  max_depth:    {row['max_depth']}")
    print(f"  Features ({len(eval(row['features']))}): {eval(row['features'])}")


══════════════════════════════════════════════════════
TOP FINE-TUNED MODEL BY RECALL
══════════════════════════════════════════════════════

Rank 1
  Recall:       0.8878
  Precision:    0.2254
  F1:           0.3595
  n_estimators: 100
  max_depth:    5.0
  Features (13): ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']


In [45]:
fine_tuned = fine_tuned.sort_values('f1', ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print("TOP FINE-TUNED MODELS BY F1")
print("══════════════════════════════════════════════════════")

for i, row in fine_tuned.head(1).iterrows():
    print(f"\nRank {i+1}")
    print(f"  F1:           {row['f1']:.4f}")
    print(f"  Recall:       {row['recall']:.4f}")
    print(f"  Precision:    {row['precision']:.4f}")
    print(f"  n_estimators: {row['n_estimators']}")
    print(f"  max_depth:    {row['max_depth']}")
    print(f"  Features ({len(eval(row['features']))}): {eval(row['features'])}")


══════════════════════════════════════════════════════
TOP FINE-TUNED MODELS BY F1
══════════════════════════════════════════════════════

Rank 1
  F1:           0.8778
  Recall:       0.8061
  Precision:    0.9634
  n_estimators: 500
  max_depth:    nan
  Features (13): ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']


In [46]:
fine_tuned = fine_tuned.sort_values('precision', ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print("TOP FINE-TUNED MODELS BY precision")
print("══════════════════════════════════════════════════════")

for i, row in fine_tuned.head(1).iterrows():
    print(f"\nRank {i+1}")
    print(f"  Precision:    {row['precision']:.4f}")
    print(f"  Recall:       {row['recall']:.4f}")
    print(f"  F1:           {row['f1']:.4f}")
    print(f"  n_estimators: {row['n_estimators']}")
    print(f"  max_depth:    {row['max_depth']}")
    print(f"  Features ({len(eval(row['features']))}): {eval(row['features'])}")


══════════════════════════════════════════════════════
TOP FINE-TUNED MODELS BY precision
══════════════════════════════════════════════════════

Rank 1
  Precision:    0.9634
  Recall:       0.8061
  F1:           0.8778
  n_estimators: 500
  max_depth:    nan
  Features (13): ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']
